# Notebook 06: Live Trading Setup Guide

This notebook walks through every step needed to move from data to live factor-based signals.
No backtest simulation here — just the operational pipeline.

**Workflow:**
1. Load or fetch data
2. Compute adaptive factor composite
3. Monitor factor health
4. Generate trade signals
5. (Optional) Run the Streamlit dashboard

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from pathlib import Path
from loguru import logger

logger.remove()
logger.add(lambda msg: print(msg, end=''), colorize=False, format='{time:HH:mm:ss} | {level} | {message}')

print('✅ Imports ready')

## Step 1: Load Data

For live use, call `YFinanceFetcher` with `force_refresh=True`. Here we use cached demo data.

In [ ]:
raw_dir = Path('../data/raw')
parquet_files = sorted(raw_dir.glob('*.parquet'))
data = pd.read_parquet(parquet_files[0])

print(f'Data shape: {data.shape}')
print(f'Tickers: {data.index.get_level_values("ticker").unique().tolist()}')
print(f'Date range: {data.index.get_level_values("date").min()} → {data.index.get_level_values("date").max()}')

## Step 2: Initialize Factors and Adaptive Manager

Configure with your chosen parameters. For production, load from `outputs/optimization/best_params.json`.

In [ ]:
import json
from src.factors.momentum import MomentumFactor
from src.factors.mean_reversion import MeanReversionFactor
from src.factors.volatility import VolatilityFactor
from src.factors.adaptive_composite import AdaptiveCompositeFactor, AdaptiveCompositeManager

best_params_path = Path('../outputs/optimization/best_params.json')
if best_params_path.exists():
    with open(best_params_path) as f:
        params = json.load(f)
    print(f'Loaded optimized params: {params}')
else:
    params = {
        'ic_window': 63,
        'decay_halflife': 21,
        'min_weight': 0.05,
        'normalization_method': 'softmax',
        'forward_period': 21,
    }
    print(f'Using default params: {params}')

factors = [
    MomentumFactor(lookback=126, name='Momentum'),
    MeanReversionFactor(lookback=21, name='MeanReversion'),
    VolatilityFactor(window=63, name='Volatility'),
]

af = AdaptiveCompositeFactor(
    factors=factors,
    ic_window=params['ic_window'],
    decay_halflife=params['decay_halflife'],
    min_weight=params['min_weight'],
    normalization_method=params['normalization_method'],
)
manager = AdaptiveCompositeManager(af, forward_period=params['forward_period'])
print('✅ AdaptiveCompositeManager ready')

## Step 3: Set Up FactorMonitor

The monitor tracks ICIR, weight concentration (HHI), and stale-weight alerts.
Persist to SQLite for cross-session history.

In [ ]:
from src.monitoring import FactorMonitor

DB_PATH = '../outputs/factor_monitor.db'
monitor = FactorMonitor(
    manager,
    icir_threshold=0.3,
    hhi_threshold=0.5,
    db_path=DB_PATH,
)
print(f'✅ FactorMonitor ready (DB: {DB_PATH})')

## Step 4: Process New Data and Generate Signals

In a production loop you call this daily (or at your chosen rebalance frequency).

In [ ]:
all_dates = sorted(data.index.get_level_values('date').unique())

recent_dates = all_dates[-90:]
print(f'Processing {len(recent_dates)} dates (demo)...\n')

composite = None
weights = {}

for date in recent_dates:
    date = pd.Timestamp(date)
    data_slice = data[data.index.get_level_values('date') <= date]
    prices_wide = data['close'].unstack('ticker')
    prices_today = prices_wide[prices_wide.index <= date].iloc[-1]

    try:
        composite, weights = manager.compute_factor_with_update(data_slice, prices_today, date)
        metrics = monitor.check(date)
    except Exception as e:
        logger.warning(f'Skipping {date}: {e}')
        continue

if composite is not None:
    latest_date = pd.Timestamp(recent_dates[-1])
    composite_today = composite[composite.index.get_level_values('date') == latest_date]
    print(f'\n--- Factor Signals as of {latest_date.date()} ---')
    print(composite_today.droplevel('date').sort_values(ascending=False).to_string())
    print(f'\n--- Current Weights ---')
    for k, v in weights.items():
        print(f'  {k}: {v:.3f} ({v*100:.1f}%)')

## Step 5: Generate Trade List

Rank stocks by composite factor score → top quartile = BUY, rest = flat (long-only).

In [ ]:
if composite is not None:
    latest_date = pd.Timestamp(recent_dates[-1])
    composite_today = composite[composite.index.get_level_values('date') == latest_date]
    signals = composite_today.droplevel('date').dropna()
    n = len(signals)
    n_long = max(1, n // 4)

    ranked = signals.rank(ascending=False)
    buy_list = ranked[ranked <= n_long].index.tolist()
    flat_list = ranked[ranked > n_long].index.tolist()

    print('TRADE LIST:')
    print(f'  BUY  ({n_long} stocks): {buy_list}')
    print(f'  FLAT ({len(flat_list)} stocks): {flat_list}')
    print('\nTarget weights:')
    for t in buy_list:
        print(f'  {t}: {1/n_long:.3f} ({100/n_long:.1f}%)')

## Step 6: Health Summary and Dashboard

Print the monitor summary. For a web dashboard, run:

```bash
streamlit run dashboard/app.py
```

In [ ]:
monitor.summary()
print('\n✅ Live trading setup complete.')
print('   Run "streamlit run dashboard/app.py" for the real-time dashboard.')